# D4 numor analysis in Jupyter — examples

This notebook shows how to use **`d4numor.py`**, the version of *D4NumorGUI* that runs without a graphical interface. Everything the GUI does is available here as Python calls:

| GUI | Here |
|---|---|
| *Browse* + *Open Numors* | `d4.NumorSet.from_directory(dir, "416440-416455")` |
| Variable tree | `ns.list_variables()` |
| Select variables → weighted averages + table | `ns.print_table([...])` |
| Plot, *Normalize* | `ns.plot([...], normalize=True)` |
| *Binary* (first = Y, second = X) | `ns.plot_binary(y, x)` |
| Detector Movement Information | `ns.motion_summary()`, `ns.motion_table()` |
| *Export Plot* / *Export Table* | `filename="..."` / `ns.export_csv(...)` |
| Scan angles + Preset + *Calculate* | `ns.preset(30, angles)` |

**Requirements:** `numpy`, `matplotlib`, `h5py` (for `nx_reader_D4.py`), optionally `pandas`.
Use **`nx_reader_D4.py` version 2** (October 2026): the previous version cannot read the current D4 files.
`d4numor.py` and `nx_reader_D4.py` must be in the same folder as this notebook (or set `CODE_DIR` below).

Each section can be run on its own once sections 1 and 2 have been run. See `MANUAL.md` for the full reference.

## 1. Setup

Edit the three variables of the next cell: where the code is, where the raw data are and which numors to read.

In [ ]:
import os, sys

# --- Folder containing d4numor.py and nx_reader_D4.py -----------------------
CODE_DIR = "."                    # e.g. "/home/user/d4tools"

# --- Raw data and numors to analyse ------------------------------------------
RAWDATA  = "../rawdata"           # folder with the 6-digit .nxs files
NUMORS   = "499980-500029"        # ranges and lists: "499980-499989,500010", "499980-500029:2"

sys.path.insert(0, os.path.abspath(CODE_DIR))

import numpy as np
import matplotlib.pyplot as plt
import d4numor as d4

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
print("d4numor version", d4.__version__)

## 2. Loading numors

`NumorSet` reads the files once (with `nx_reader_D4.extract_multiple_variables`) and keeps every variable in memory. Three ways to select the files:

In [ ]:
# (a) Directory + numor specification  (the usual way)
ns = d4.NumorSet.from_directory(RAWDATA, NUMORS)
ns

In [ ]:
# (b) Explicit files or glob patterns
#     ns = d4.NumorSet.from_files(f"{RAWDATA}/4999*.nxs")
#     ns = d4.NumorSet.from_files([f"{RAWDATA}/499980.nxs", f"{RAWDATA}/499985.nxs"])
#
# (c) Every .nxs file of a directory
#     ns = d4.NumorSet.from_directory(RAWDATA)

# How a numor specification is expanded:
print(d4.parse_numor_list("499980-499984, 499990, 500000-500006:3"))

Notes
* Numors that do not exist are reported and skipped; numors that cannot be read are reported and removed from the set.
* Files are named `{:06d}.nxs` by default. For another naming scheme use `pattern=`, e.g. `pattern="D4_{:06d}.nxs"`.
* `verbose=False` makes the loading silent.

## 2b. Checking that the files are read correctly

`nx_reader_D4` (version 2) has two diagnostic functions. Use them on one numor whenever you suspect the NeXus format has changed:

* `check_file(path)` — where each quantity comes from (`scan:...` = scanned-variables table, a dataset path, `computed`) and what is missing;
* `list_datasets(path)` — every dataset of the file with its type, shape and first values.

In [ ]:
import nx_reader_D4 as R
print("nx_reader_D4 version", getattr(R, "__version__", "1 (old: cannot read the current files)"))

first_file = ns.filepaths[0]
R.check_file(first_file);

In [ ]:
rows = R.list_datasets(first_file, print_out=False)
for path, dtype, shape, values in rows:
    if path.startswith("entry0/sample") or "scanned_variables" in path:
        print(f"{path:60s} {dtype:8s} {shape:10s} {values}")

**About temperatures in the current files:** `SampleTemp` (scanned variable / `entry0/sample/temperature`) can be a constant if no sensor is connected to that channel, while `RegTemp` (`regulation_temperature`) and `SetPTemp` (`setpoint_temperature`) follow the temperature steps. Compare them before choosing which one to use:

In [ ]:
ns.print_table(["SetPTemp", "RegTemp", "SampleTemp"], averages=False)

## 3. Which variables are available?

The equivalent of the GUI tree: variables grouped by category, `num` (can be plotted) or `text`.

In [ ]:
ns.list_variables()

In [ ]:
# Also show the variables of VARIABLE_MAP that the reader did not return ([ ])
ns.list_variables(show_unavailable=True);

In [ ]:
# Direct access to the data (numpy arrays, one value per numor)
print("Numors     :", ns.numors.astype(int))
print("RegTemp    :", ns.get("RegTemp"))
print("Subtitle   :", ns.get("subtitle")[:3], "...")
print("Numeric?   :", ns.is_numeric("RegTemp"), ns.is_numeric("subtitle"))

# Names are case-insensitive and the GUI labels work too:
print(ns.resolve("sampletemp"), "|", ns.resolve("Monitor counts"))

## 4. Overview of the measurement

In [ ]:
ns.summary();

## 5. Tables and weighted averages

`print_table` gives exactly what the GUI shows in its text panel: the monitor- and detector-weighted averages of the numeric variables, then one line per numor. Text variables (subtitle, times…) can be mixed with numeric ones.

In [ ]:
ns.print_table(["Monitor", "TotalCount", "Time", "RegTemp", "subtitle"])

In [ ]:
# Only the numbers
wa = ns.weighted_averages(["RegTemp", "ttheta_value"])
for var, d in wa.items():
    print(f"{var:12s}  monitor-weighted = {d['monitor']:.4f}   detector-weighted = {d['detector']:.4f}")

# Statistics of one variable
ns.stats("RegTemp")

In [ ]:
# Weighted average with any other variable as weight
ns.weighted_average("RegTemp", weight="Time")

## 6. Plots versus numor

In [ ]:
# One variable
ns.plot("RegTemp");

In [ ]:
# Several variables normalised to their mean (GUI 'Normalize' check box):
# a quick way to check that monitor and detector follow each other
ns.plot(["Monitor", "TotalCount", "monrate", "detrate"], normalize=True,
        title="Counting stability");

In [ ]:
# One panel per variable, sharing the numor axis
ns.plot(["RegTemp", "SampleTemp", "detrate"], separate=True);

In [ ]:
# The nine detector blocks, normalised
ns.plot([f"SumDet{i}" for i in range(1, 10)], normalize=True, figsize=(9, 5));

In [ ]:
# Saving a figure (png, pdf, svg... from the extension) and customising it.
# plot() returns the matplotlib figure and axes, so anything can be changed.
fig, ax = ns.plot("RegTemp", filename="RegTemp.png")
for T in (220, 230, 240):
    ax.axhline(T, color="grey", ls="--", lw=0.8)
ax.set_ylim(210, 250)
fig

In [ ]:
# Drawing into your own axes, e.g. a 1x2 layout
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
ns.plot("Monitor", ax=a1)
ns.plot("ttheta_value", ax=a2, style="s--")
plt.show()

## 7. Binary mode: Y versus X

The first variable is Y and the second X, as in the GUI. Points are matched numor by numor.

In [ ]:
ns.plot_binary("detrate", "ttheta_value", annotate=True);
print(ns.binary_table("detrate", "ttheta_value"))

In [ ]:
# Count rate versus temperature, without lines, saved to a file
ns.plot_binary("detrate", "RegTemp", style="o", filename="rate_vs_T.png");

In [ ]:
# The matched arrays, for your own analysis (e.g. a linear fit)
numors, x, y = ns.binary_data("detrate", "ttheta_value")
p = np.polyfit(x, y, 1)
print(f"detrate = {p[0]:.2f} * 2theta + {p[1]:.2f}")

## 8. Detector movement

Computed with `nx_reader_D4.analyse_angular_motion` from the 2θ values and the start/end times. On each line, Δ2θ, time and speed refer to the movement **before** that numor (the first line is always 0).

In [ ]:
print(ns.motion_table())

In [ ]:
ns.plot(["moving_time", "angular_speed"], separate=True);

## 9. Preset calculator

The scan-angle boxes and the *Calculate* button of the GUI. The monitor rate and the detector speed are taken from the loaded numors.

* `value < 1000`: total scan time in **minutes** → result: **monitor preset per point**.
* `value ≥ 1000`: **monitor counts per point** → result: **total scan time**.

In [ ]:
angles = [-0.5, -0.5, 0.0, 4.5, 5.0, 6.0, 7.5, 9.0, 10.0, 10.75]   # GUI defaults
res = ns.preset(30, angles)          # 30 minutes in total -> monitor per point

In [ ]:
ns.preset(100000, angles);           # 100000 monitor counts per point -> total time

In [ ]:
# Several inputs at once
print(ns.preset_table([30, 60, 120, 240, 10000, 100000], angles))

In [ ]:
# Which rates are used?  The GUI uses the median monitor rate and the mean of the
# per-numor speeds (zeros included, which lowers it). Alternatives:
print(ns.rates())

ns.preset(30, angles, angular_speed="motion");      # total range / total moving time
ns.preset(30, angles, mon_rate=1700, angular_speed=0.05);   # your own values

In [ ]:
# The result is a dict, handy for scripting
res = ns.preset(60, angles, print_out=False)
print(res["preset"], res["preset_units"], f"{100*res['moving_ratio']:.1f} % moving")

## 10. Details of a single numor

Uses `nx_reader_D4.read_nexus` + `show_single_numor` (energy, wavelength, reactor cycle, detector sums…).

In [ ]:
ns.info(int(ns.numors[0]));

In [ ]:
# Any file, without loading a set
d4.numor_info(os.path.join(RAWDATA, f"{int(ns.numors[-1]):06d}.nxs"));

## 11. Exporting

In [ ]:
# CSV: one line per numor (labels as column names, like the GUI)
ns.export_csv(["Monitor", "TotalCount", "Time", "RegTemp", "subtitle"], "selection.csv")

# Every variable, internal names as column names
ns.export_csv(None, "all_variables.csv", labels=False)

# Binary mode: Numor, X, Y
ns.export_binary_csv("detrate", "RegTemp", "rate_vs_T.csv")

In [ ]:
# pandas DataFrame indexed by numor (needs pandas)
df = ns.to_dataframe(["ttheta_value", "Monitor", "TotalCount", "RegTemp", "detrate", "subtitle"])
df.head()

In [ ]:
# ...which makes grouping easy, e.g. statistics per sample
df.groupby("subtitle")[["RegTemp", "detrate"]].agg(["mean", "std", "count"])

## 12. Going further

### 12.1 Reading more variables
`NumorSet` asks the reader for the same variables as the GUI. Any other attribute created by `nx_reader_D4.read_nexus` (`ttheta_target`, `z_value`, `RegTemp2`, `SetPTemp2`, `SCrot_value`, `VolGams_value`, `detdata`, …) can be added with `extra_variables`.

In [ ]:
ns2 = d4.NumorSet.from_directory(RAWDATA, NUMORS,
                                 extra_variables=["ttheta_target", "z_value", "RegTemp2", "detdata"],
                                 verbose=False)
ns2.print_table(["ttheta_target", "ttheta_value", "RegTemp", "RegTemp2"], averages=False)

In [ ]:
# 'detdata' holds one array (all detection cells) per numor
detdata = np.vstack(ns2.get("detdata"))
print("shape (numors, cells):", detdata.shape)

plt.figure(figsize=(9, 4))
plt.imshow(detdata, aspect="auto", interpolation="nearest")
plt.colorbar(label="counts"); plt.xlabel("cell"); plt.ylabel("numor index")
plt.show()

### 12.2 Nice labels for new variables
Labels and categories come from `d4.VARIABLE_MAP` (the same dictionary as in the GUI). Add entries to it before loading:

In [ ]:
d4.VARIABLE_MAP["detdata"] = {"label": "Detector cells", "category": "Detector", "tooltip": "576 cells"}
ns2.list_variables();         # 'detdata' now appears under "Detector"

### 12.3 Comparing two sets of numors

In [ ]:
first, last = int(ns.numors[0]), int(ns.numors[-1])
mid = (first + last) // 2
a = d4.NumorSet.from_directory(RAWDATA, f"{first}-{mid}", verbose=False)
b = d4.NumorSet.from_directory(RAWDATA, f"{mid+1}-{last}", verbose=False)

fig, ax = plt.subplots()
a.plot_binary("detrate", "ttheta_value", ax=ax, style="o-")
b.plot_binary("detrate", "ttheta_value", ax=ax, style="s--")
ax.legend([f"{first}-{mid}", f"{mid+1}-{last}"]); ax.set_title("Two scans")
plt.show()

### 12.4 Using the reader directly
The module of the reader is available as `ns.reader` (it is `nx_reader_D4`).

In [ ]:
nx = ns.reader.read_nexus(ns.filepaths[0])
print(nx.RegTemp, nx.cycle, nx.reactor, nx.power, "MW")

## 13. Command-line version
The same file is a command-line program (useful over `ssh`). Figures are written to files, no display needed. In a notebook the commands can be run with `!`:

In [ ]:
!python {CODE_DIR}/d4numor.py --help

In [ ]:
!python {CODE_DIR}/d4numor.py summary -d {RAWDATA} -n {NUMORS} -q --reader-dir {CODE_DIR}